# Absolute Effectiveness — Africa Cropland Test
Compute habitat condition and loss metrics for a selected African PA, using the
African Cropland Dataset (AFCD) for cropland detection (`*_africa` functions).

In [ ]:
# Select site by WDPAID (one from the list of 30 in variables.py)
site_id = 2017

In [ ]:
from pathlib import Path
import os
import sys
import ee
import geemap

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    ANALYSIS_END_YR,
    GLC_LABELS,
    DRIVER_LABELS,
)

from absolute_effectiveness.site_selector import SiteSelector
from absolute_effectiveness.data_processor import DataProcessor
from absolute_effectiveness.habitat_condition import HabitatConditionAnalyzer
from absolute_effectiveness.habitat_loss import HabitatLossAnalyzer
from absolute_effectiveness.visualization import VisualizationService

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()
processor = DataProcessor.from_africa_defaults()
condition_analyzer = HabitatConditionAnalyzer()
loss_analyzer = HabitatLossAnalyzer()
visualization_service = VisualizationService()

In [ ]:
# Get test sites and site-specific information
test_sites = site_selector.get_test_sites()
START_YR = site_selector.set_start_yr(test_sites, site_id)
site_geom = site_selector.get_site_geom(test_sites, site_id)

# Check if PA designation year is valid for analysis
site_selector.check_start_yr(START_YR)

In [ ]:
# Process land-masked input datasets (extent, loss, and site area)
GLC_processed = processor.process_glc(test_sites, START_YR)
GPW_processed = processor.process_gpw(START_YR)
NFW_processed = processor.process_nfw(test_sites)
HGFC_processed = processor.process_hgfc(START_YR)
AFCD_processed = processor.process_afcd(START_YR)
land_mask = processor.get_land_mask()

# Build land-masked habitat raster for extent scoring
habitat_raster = condition_analyzer.get_habitat_raster_africa(
    GLC_processed, HGFC_processed, GPW_processed, NFW_processed, AFCD_processed
)

In [ ]:
habitat_extent_score = condition_analyzer.calc_habitat_extent_score(
    habitat_raster, site_geom, land_mask
)

# Build unmasked habitat raster for intactness (water excluded inside kernel, not zeroed)
GLC_intactness = processor.process_glc(test_sites, START_YR, land_masked=False)
GPW_intactness = processor.process_gpw(START_YR, land_masked=False)
NFW_intactness = processor.process_nfw(test_sites, land_masked=False)
HGFC_intactness = processor.process_hgfc(START_YR, land_masked=False)
AFCD_intactness = processor.process_afcd(START_YR, land_masked=False)
habitat_raster_intactness = condition_analyzer.get_habitat_raster_africa(
    GLC_intactness, HGFC_intactness, GPW_intactness, NFW_intactness, AFCD_intactness
)

# Calculate Habitat Intactness score
exp_kernel = condition_analyzer.build_kernel()
intactness_raster = condition_analyzer.get_intactness_raster(
    habitat_raster_intactness, site_geom, exp_kernel, land_mask
)
habitat_intactness_score = condition_analyzer.calc_intactness_score(
    intactness_raster, site_geom
)

# Calculate overall Habitat Condition score
habitat_condition_score = condition_analyzer.calc_habitat_condition_score(
    habitat_extent_score, habitat_intactness_score
)

In [ ]:
# Calculate Habitat Loss score (AFCD-based cropland detection)
habitat_loss_raster, habitat_start_raster = loss_analyzer.get_habitat_loss_raster_africa(
    GLC_processed, GPW_processed, HGFC_processed, AFCD_processed, START_YR
)
(
    habitat_loss_area_km2,
    habitat_start_area_km2,
    site_area_km2,
    habitat_loss_pct_site,
    habitat_loss_pct_start,
    habitat_loss_score,
) = loss_analyzer.calc_overall_habitat_loss(
    habitat_loss_raster, habitat_start_raster, site_geom, land_mask
)

# Analyze drivers and types of habitat loss
driver_class = loss_analyzer.get_driver_class_image_africa(
    GLC_processed, GPW_processed, AFCD_processed, habitat_loss_raster
)
driver_dict = loss_analyzer.calc_class_area_and_pct(
    driver_class, site_geom, site_area_km2, habitat_start_area_km2
)
habitat_class = loss_analyzer.get_habitat_class_image(
    GLC_processed, habitat_loss_raster, START_YR
)
habitat_dict = loss_analyzer.calc_habitat_class_area_and_pct(
    habitat_class,
    habitat_start_raster,
    GLC_processed,
    START_YR,
    site_geom,
)

# Create Sentinel-2 composites (for visualization purposes only)
start_s2_composite = visualization_service.get_s2_med_composite(site_geom, START_YR)
end_s2_composite = visualization_service.get_s2_med_composite(
    site_geom, ANALYSIS_END_YR
)

In [ ]:
# Print all results
print(f"Site ID: {site_id}")
print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
print(f"\nHabitat Extent Score: {habitat_extent_score:.2f}")
print(f"Habitat Intactness Score: {habitat_intactness_score:.2f}")
print(f"\nHabitat Condition Score: {habitat_condition_score:.10f}")
print(f"Habitat Loss Score: {habitat_loss_score:.10f}")
print("\nOverall habitat loss:")
loss_analyzer.print_habitat_loss_metrics(
    "Total",
    habitat_loss_area_km2,
    habitat_loss_pct_start,
    pct_site=habitat_loss_pct_site,
)
print("\nDrivers of habitat loss:")
loss_analyzer.translate_results(driver_dict, DRIVER_LABELS)
print("\nTop 4 types of habitat lost:")
loss_analyzer.translate_results(habitat_dict, GLC_LABELS, pct_reference_key="pct_class")

## Visualization

In [ ]:
from utils.variables import GLC_PALETTE, INTACTNESS_SCALE

# Sentinel-2 RGB viz parameters
s2_rgb_viz = {
    "min": 0.0,
    "max": 0.3,
    "bands": ["B4", "B3", "B2"],
}

score_palette = ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]
intactness_viz = {"min": 0, "max": 1, "palette": score_palette}

habitat_gradient = {
    31: "#f9f1d6",
    20: "#e9c157",
    21: "#d1861f",
    22: "#8fb03f",
    18: "#6b6323",
}
habitat_palette = list(GLC_PALETTE)
for class_id, color in habitat_gradient.items():
    habitat_palette[class_id - 1] = color
habitat_viz = {"min": 1, "max": 36, "palette": habitat_palette}
habitat_legend_classes = list(habitat_gradient)

# Driver classes are 1-4 (see DRIVER_LABELS), so palette position = class - 1
driver_palette = ["yellow", "red", "orange", "green"]
driver_viz = {"min": 1, "max": 4, "palette": driver_palette}
# Display order for the legend, largest driver first
driver_legend_order = [4, 3, 1, 2]

def habitat_color(class_id):
    """Map color for a GLC class in the Habitat Extent layer."""
    return habitat_palette[class_id - 1]

def glc_label(class_id):
    """GLC class label without the fractional-cover qualifier, e.g. "(fc<0.15)"."""
    return GLC_LABELS[class_id].split(" (")[0]

def ramp_legend_html(title, palette, vmin=0, vmax=1, ticks=(0, 0.5, 1), width=200):
    """Compact continuous-ramp legend, styled to match the palette of a score layer."""
    stops = ", ".join(
        f"{color} {100 * i / (len(palette) - 1):.1f}%" for i, color in enumerate(palette)
    )
    tick_labels = "".join(
        f'<span>{vmin + t * (vmax - vmin):g}</span>' for t in ticks
    )
    return f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,sans-serif;
                background:rgba(255,255,255,0.94);border-radius:5px;
                padding:9px 11px 8px 11px;color:#1a1c1e;">
      <div style="font-size:14px;font-weight:600;letter-spacing:0.01em;margin-bottom:7px;">
        {title}
      </div>
      <div style="width:{width}px;height:11px;border-radius:2px;
                  background:linear-gradient(to right, {stops});
                  box-shadow:inset 0 0 0 1px rgba(0,0,0,0.18);"></div>
      <div style="display:flex;justify-content:space-between;width:{width}px;
                  font-size:12px;color:#5b6066;margin-top:4px;
                  font-variant-numeric:tabular-nums;">
        {tick_labels}
      </div>
    </div>
    """

def swatch_legend_html(title, entries, swatch_px=13):
    """Categorical legend: one color swatch per class. Matches ramp_legend_html styling.

    Args:
        entries: Sequence of (label, color) pairs, in the order they should appear.
    """
    rows = "".join(
        f'<div style="display:flex;align-items:center;gap:7px;margin-top:5px;">'
        f'<span style="width:{swatch_px}px;height:{swatch_px}px;flex:0 0 auto;'
        f'border-radius:2px;background:{color};'
        f'box-shadow:inset 0 0 0 1px rgba(0,0,0,0.18);"></span>'
        f'<span style="font-size:12px;color:#3a3f45;white-space:nowrap;">{label}</span>'
        f"</div>"
        for label, color in entries
    )
    return f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,sans-serif;
                background:rgba(255,255,255,0.94);border-radius:5px;
                padding:9px 11px 9px 11px;color:#1a1c1e;">
      <div style="font-size:14px;font-weight:600;letter-spacing:0.01em;margin-bottom:2px;">
        {title}
      </div>
      {rows}
    </div>
    """

Map = geemap.Map(height=900, zoom_snap=0)
Map.add_basemap("CartoDB.DarkMatter")
# Map.add_basemap("CartoDB.Positron")
Map.add_basemap("Esri.WorldImagery")

# Map.addLayer(start_s2_composite, s2_rgb_viz, "S2, Start Year", 0)
# Map.addLayer(end_s2_composite, s2_rgb_viz, "S2, End Year", 0)
Map.addLayer(GLC_processed.select("GLC_2022"), {"min": 0, "max": 36, "palette": GLC_PALETTE}, "Global Land Cover", 0)
Map.addLayer(GPW_processed.select("GPW_2022").selfMask(), {"min": 1, "max": 2, "palette": ["#ffcd73", "#ff9916"]}, "Global Pasture Watch", 0)
Map.addLayer(NFW_processed.selfMask(), {"palette": "teal"}, "Natural Forests of the World", 0)
Map.addLayer(HGFC_processed.selfMask(), {"min": START_YR - 2000, "max": ANALYSIS_END_YR - 2000, "palette": ["yellow", "red"]}, "Hansen Global Forest Change", 0)
Map.addLayer(AFCD_processed.select("AFCD_2022").selfMask(), {"min": 0, "max": 1, "palette": "yellow"}, "African Cropland 2022", 0)
Map.addLayer(site_geom, {"color": "white"}, "Test site", 1, 0.5)
Map.addLayer(habitat_raster.clip(site_geom), habitat_viz, "Habitat Extent")
Map.addLayer(
    intactness_raster.clip(site_geom).reproject(
        crs=habitat_raster.projection(), scale=INTACTNESS_SCALE
    ),
    intactness_viz,
    "Habitat Intactness",
)
# Map.addLayer(habitat_start_raster.selfMask().clip(site_geom), {'palette': GLC_PALETTE}, "Habitat at start year")
# Map.addLayer(habitat_loss_raster.selfMask().clip(site_geom), {'palette': ['red']}, "Habitat Loss")
Map.addLayer(driver_class.selfMask().clip(site_geom), driver_viz, "Drivers of habitat loss", 0)
Map.addLayer(habitat_class.selfMask().clip(site_geom), {"palette": GLC_PALETTE}, "Types of habitat lost", 0)

# # Legend for the Habitat Intactness layer (0 = least intact, 1 = most intact)
# Map.add_html(
#     ramp_legend_html(
#         "Habitat Intactness", score_palette, intactness_viz["min"], intactness_viz["max"]
#     ),
#     position="bottomright",
# )

# # Legend for the habitat types present in the Habitat Extent layer
# Map.add_html(
#     swatch_legend_html(
#         "Habitat Types",
#         [
#             (glc_label(class_id), habitat_color(class_id))
#             for class_id in habitat_legend_classes
#         ],
#     ),
#     position="bottomright",
# )

# Legend for the Drivers of habitat loss layer
Map.add_html(
    swatch_legend_html(
        "Drivers of Habitat Loss",
        [
            (DRIVER_LABELS[class_id], driver_palette[class_id - 1])
            for class_id in driver_legend_order
        ],
    ),
    position="bottomright",
)

Map.centerObject(site_geom)

Map